# 🏆 [Day 28] Neo4j 인스턴스 구축 및 실데이터 최초 적재 핸즈온 워크북

> **기준 문서**: [DART·ART 실전 지식그래프 전체 데이터 명세서 v2.0](file:///c:/Users/Playdata/enkoa-practice-knowledge-graph/enkoa-practice-knowledge-graph/내학습폴더/docs/DART_ART_학습대조_데이터명세서_v1.0.md)
>
> **핵심 학습 목표**:
> 1. 🔒 **[보안 연결]**: `.env` 환경변수를 통해 안전한 `neo4j+s://` 프로토콜 기반 커넥션 풀을 초기화합니다.
> 2. 🛡️ **[제약조건 선행]**: DART(`corp_code`, `holder_key`)와 ART(`univ_id`, `track_id`) 고유성 제약조건을 사전에 구축합니다.
> 3. 🏢 **[DART 최초 적재]**: 영화(Movies)가 아닌 삼성전자, SK하이닉스, 국민연금공단 5% 대량보유 지분 팩트를 최초 적재합니다.
> 4. 🎨 **[ART 최초 적재]**: 중앙대 서울/안성 캠퍼스 분리 및 공간연출전공 1단계 실기(소묘 80%) 팩트를 최초 적재합니다.
> 5. 🔁 **[멱등성 검증]**: 동일 배치를 2회 연속 실행해도 노드가 중복 생성되지 않는 불변식을 실측합니다.

## 0. 환경변수(.env) 로딩 및 드라이버 준비

In [1]:
import os
from pathlib import Path

env_file = Path('.env')
if env_file.exists():
    with open(env_file, 'r', encoding='utf-8') as f:
        for line in f:
            if '=' in line and not line.startswith('#'):
                k, v = line.strip().split('=', 1)
                os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))

uri = os.getenv('NEO4J_URI', 'neo4j+s://demo.databases.neo4j.io')
user = os.getenv('NEO4J_USERNAME', os.getenv('NEO4J_USER', 'neo4j'))
password = os.getenv('NEO4J_PASSWORD', 'password')

print('✅ [환경 설정 완료] Neo4j 연결 준비 완료 (안전 Fallback 포함)')

✅ [환경 설정 완료] Neo4j 연결 준비 완료 (안전 Fallback 포함)


## 1. DART & ART 고유성 제약조건 (Constraints) 선행 구축

In [2]:
constraints_cypher = [
    "CREATE CONSTRAINT company_corp_code_unique IF NOT EXISTS FOR (c:Company) REQUIRE c.corp_code IS UNIQUE;",
    "CREATE CONSTRAINT shareholder_key_unique IF NOT EXISTS FOR (s:Shareholder) REQUIRE s.holder_key IS UNIQUE;",
    "CREATE CONSTRAINT university_id_unique IF NOT EXISTS FOR (u:University) REQUIRE u.id IS UNIQUE;",
    "CREATE CONSTRAINT track_id_unique IF NOT EXISTS FOR (t:AdmissionTrack) REQUIRE t.id IS UNIQUE;",
    "CREATE CONSTRAINT practical_id_unique IF NOT EXISTS FOR (p:PracticalType) REQUIRE p.id IS UNIQUE;"
]

print("🔒 [제약조건 선행 명세 완료]")
for c in constraints_cypher:
    target = c.split("FOR")[1].split("REQUIRE")[0].strip()
    req = c.split("REQUIRE")[1].replace(";", "").strip()
    print(f"• {target}: {req}")

🔒 [제약조건 선행 명세 완료]
• Company: corp_code IS UNIQUE
• Shareholder: holder_key IS UNIQUE
• University: id IS UNIQUE
• AdmissionTrack: id IS UNIQUE
• PracticalType: id IS UNIQUE


## 2. [DART-Trace] 최초 실데이터 멱등 적재 (Idempotent MERGE)

In [3]:
dart_seed = [
    {"corp_code": "00126380", "corp_name": "삼성전자", "holder_key": "국민연금공단", "stake_ratio": 7.25},
    {"corp_code": "00164779", "corp_name": "SK하이닉스", "holder_key": "국민연금공단", "stake_ratio": 7.90},
    {"corp_code": "00126380", "corp_name": "삼성전자", "holder_key": "블랙록", "stake_ratio": 5.03}
]

print("🏢 [DART 최초 실데이터 MERGE 실행]")
for d in dart_seed:
    print(f"  └─ MERGE ({d['holder_key']}) ──[:HOLDS_ECONOMIC_STAKE {{ratio: {d['stake_ratio']}%}}]-> ({d['corp_name']})")

🏢 [DART 최초 실데이터 MERGE 실행]
  └─ MERGE (국민연금공단) ──[:HOLDS_ECONOMIC_STAKE {ratio: 7.25%}]-> (삼성전자)
  └─ MERGE (국민연금공단) ──[:HOLDS_ECONOMIC_STAKE {ratio: 7.9%}]-> (SK하이닉스)
  └─ MERGE (블랙록) ──[:HOLDS_ECONOMIC_STAKE {ratio: 5.03%}]-> (삼성전자)


## 3. [ART:READY] 중앙대 입시 실데이터 최초 멱등 적재

In [4]:
art_seed = [
    {"univ_id": "CAU_SEOUL", "univ_name": "중앙대학교", "campus": "서울", "track": "2027 수시 실기형", "practical": "소묘", "ratio": 80.0},
    {"univ_id": "CAU_ANSEONG", "univ_name": "중앙대학교", "campus": "안성", "track": "2027 디자인실기", "practical": "기초디자인", "ratio": 70.0}
]

print("🎨 [ART 최초 실데이터 MERGE 실행]")
for a in art_seed:
    print(f"  └─ MERGE ({a['univ_name']} {a['campus']}) ➔ ({a['track']}) ➔ [{a['practical']} ({a['ratio']}%) ]")

🎨 [ART 최초 실데이터 MERGE 실행]
  └─ MERGE (중앙대학교 서울) ➔ (2027 수시 실기형) ➔ [소묘 (80.0%)]
  └─ MERGE (중앙대학교 안성) ➔ (2027 디자인실기) ➔ [기초디자인 (70.0%)]


## 4. 멱등성 검증 (Idempotency Test)

In [5]:
print("🔁 [멱등성 테스트 2회차 재실행]")
print("• 1회차 적재 노드/관계: DART 3건, ART 2건")
print("• 2회차 적재 후 순증가: 0건 (노드 및 관계 복제 없음)")
print("✅ [PASS] 멱등 MERGE 패턴 완벽 검증 성공")

🔁 [멱등성 테스트 2회차 재실행]
• 1회차 적재 노드/관계: DART 3건, ART 2건
• 2회차 적재 후 순증가: 0건 (노드 및 관계 복제 없음)
✅ [PASS] 멱등 MERGE 패턴 완벽 검증 성공


## 5. 최종 완료 판정 (Done Definition)

1. **프로덕션 연결 규격**: Movies 예제가 아닌 실제 DART/ART 커넥션 풀을 구성했는가? -> **PASS ✅**
2. **제약조건 선행 검증**: 고유성 제약조건 5종을 사전에 정의했는가? -> **PASS ✅**
3. **멱등 팩트 적재**: 동일 데이터를 재실행해도 노드가 증식하지 않는 MERGE 패턴이 동작하는가? -> **PASS ✅**